# Chapter 9 §9.5: Fashion Video Generator with Evaluator-Optimizer Pattern

This notebook accompanies Chapter 9 §9.5 of *Context Engineering with DSPy*: **Fashion Video Generator with Evaluator-Optimizer Pattern**.

**Required environment variables (set in your `.env`):**

- `OPENAI_API_KEY`
- `FAL_KEY`

**Service dependency:** This notebook is designed to be paired with [fal.ai](https://fal.ai/) for real image (Flux) and video (Kling) generation. The shipped version uses placeholder `generate_image` and `generate_video` functions that return fake URLs so the notebook runs without `FAL_KEY` (and avoids $2-5 per run in real image/video costs). To wire in real generation, replace the placeholders with `fal_client.submit(...)` calls as shown in each function's docstring.

## Overview

```
9.5 Fashion Video Generator with Evaluator-Optimizer Pattern

Demonstrates three patterns for multimodal pipelines:
1. Tool Loadout — wrapping external image/video APIs as DSPy tools
2. Multi-stage pipeline — text -> image -> video with quality checks
3. Evaluator-Optimizer — dspy.Refine generates, evaluates, and iterates

The core problem is Context Clash: the image might match the prompt perfectly,
but the video model introduces artifacts. Quality regressions between stages
are invisible without evaluation. dspy.Refine catches them.

Failure mode: Context Clash (conflicting quality signals between stages)
Technique: Tool Loadout + Control Flow (multi-stage pipeline)
Agentic pattern: Evaluator-Optimizer
Key DSPy feature: dspy.Refine, multimodal types

Note: Uses placeholder functions for image/video generation to avoid $2-5/run
API costs. In production, replace with real fal.ai/Replicate/Stability calls.

Usage:
    python video_generator.py                                  # full demo
    python video_generator.py --concept "cyberpunk streetwear"  # custom concept
    python video_generator.py --skip-refine                    # baseline only
```


In [ ]:
%pip install -r ../requirements.txt -q

## Imports and LM configuration

In [ ]:
import os
import hashlib
from dotenv import load_dotenv

import dspy

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
lm = dspy.LM("openai/gpt-4o-mini", api_key=api_key)
dspy.configure(lm=lm)

## Environment check (fal.ai)

Real image/video generation requires `FAL_KEY`. The shipped placeholders return fake URLs and incur zero spend — this cell prints which path you're on.

In [ ]:
if not os.getenv("FAL_KEY"):
    print("FAL_KEY not set — using placeholder generate_image / generate_video (fake URLs, no spend).")
else:
    print("FAL_KEY found — you can swap the placeholders for real fal_client.submit() calls.")

## Simulated external tools (replace with real APIs in production)

In [ ]:
def generate_image(prompt: str, style: str = "photorealistic") -> str:
    """Generate an image from a text prompt.

    In production, replace with:
        import fal_client
        result = fal_client.submit("fal-ai/flux/dev", arguments={
            "prompt": prompt,
            "image_size": "portrait_4_3",
            "num_inference_steps": 28,
        })
        return result.get()["images"][0]["url"]

    Cost: ~$0.05-0.15 per image (Flux)

    Args:
        prompt: Detailed image generation prompt
        style: Visual style guidance

    Returns:
        URL of the generated image
    """
    h = hashlib.md5(prompt.encode()).hexdigest()[:8]
    return f"https://placeholder.example.com/image_{h}.jpg"

def generate_video(image_url: str, motion_prompt: str, duration: int = 5) -> str:
    """Convert a still image into a video with motion.

    In production, replace with:
        import fal_client
        result = fal_client.submit(
            "fal-ai/kling-video/v1/standard/image-to-video",
            arguments={
                "image_url": image_url,
                "prompt": motion_prompt,
                "duration": str(duration),
                "aspect_ratio": "16:9",
            },
        )
        return result.get()["video"]["url"]

    Cost: ~$0.50-2.00 per video (Kling)

    Args:
        image_url: URL of the source image
        motion_prompt: Description of desired motion/animation
        duration: Video duration in seconds

    Returns:
        URL of the generated video
    """
    h = hashlib.md5((image_url + motion_prompt).encode()).hexdigest()[:8]
    return f"https://placeholder.example.com/video_{h}.mp4"

## Signatures

In [ ]:
class GenerateImagePrompt(dspy.Signature):
    """Create a detailed image generation prompt for a fashion concept.

    Good prompts include: lighting, camera angle, background, fabric texture,
    color palette, model pose, and photographic style. Be specific enough
    that the image generation model produces exactly what the brand needs.
    """

    concept: str = dspy.InputField(desc="Fashion concept (e.g., 'sustainable streetwear with earthy tones')")
    target_audience: str = dspy.InputField(desc="Target demographic for the video")
    image_prompt: str = dspy.OutputField(
        desc="Detailed Flux/DALL-E prompt: lighting, angle, background, "
             "fabric, colors, pose, photographic style"
    )
    motion_direction: str = dspy.OutputField(
        desc="Suggested camera motion for the video stage "
             "(e.g., 'slow dolly in', 'smooth pan left to right')"
    )

class EvaluateVideoQuality(dspy.Signature):
    """Evaluate the quality of a fashion video generation pipeline output.

    Score each dimension independently. A video can have perfect brand fit
    but poor technical quality, or vice versa. Don't let one dimension
    influence another.
    """

    concept: str = dspy.InputField(desc="Original fashion concept")
    target_audience: str = dspy.InputField(desc="Target demographic")
    image_prompt: str = dspy.InputField(desc="The prompt used to generate the image")
    motion_direction: str = dspy.InputField(desc="The motion description for the video")

    prompt_specificity: float = dspy.OutputField(
        desc="1-10: How specific and actionable is the image prompt? "
             "Penalize vague descriptions like 'beautiful fashion photo'."
    )
    brand_alignment: float = dspy.OutputField(
        desc="1-10: Does the prompt match the concept and audience? "
             "A luxury brand prompt shouldn't feel like fast fashion."
    )
    motion_coherence: float = dspy.OutputField(
        desc="1-10: Does the motion direction complement the image? "
             "Fast motion on a luxury close-up is a clash."
    )
    overall: float = dspy.OutputField(desc="1-10: Overall quality score")

## Video generation module

In [ ]:
class FashionVideoGenerator(dspy.Module):
    """Text -> Image -> Video pipeline for fashion content.

    Stage 1: LLM generates a detailed image prompt + motion direction
    Stage 2: Image generation API creates the still frame
    Stage 3: Video generation API animates the frame
    """

    def __init__(self):
        super().__init__()
        self.prompt_gen = dspy.ChainOfThought(GenerateImagePrompt)

    def forward(self, concept, target_audience):
        # Stage 1: Generate detailed prompts
        result = self.prompt_gen(concept=concept, target_audience=target_audience)

        # Stage 2: Generate image (external tool)
        image_url = generate_image(result.image_prompt)

        # Stage 3: Generate video (external tool)
        video_url = generate_video(image_url, result.motion_direction)

        return dspy.Prediction(
            image_prompt=result.image_prompt,
            motion_direction=result.motion_direction,
            image_url=image_url,
            video_url=video_url,
        )

## Reward function for dspy.Refine

In [ ]:
def video_reward(args, pred):
    """Score the video pipeline output using an LLM judge.

    This is the reward function for dspy.Refine. It evaluates the upstream
    decisions (prompt quality, motion coherence) that determine downstream
    video quality. In production with real APIs, you'd also analyze the
    actual generated image and video.

    Returns a float in [0, 1].
    """
    evaluator = dspy.Predict(EvaluateVideoQuality)

    scores = evaluator(
        concept=args["concept"],
        target_audience=args["target_audience"],
        image_prompt=pred.image_prompt,
        motion_direction=pred.motion_direction,
    )

    # Weighted combination: brand alignment matters most
    try:
        weighted = (
            0.20 * float(scores.prompt_specificity)
            + 0.35 * float(scores.brand_alignment)
            + 0.25 * float(scores.motion_coherence)
            + 0.20 * float(scores.overall)
        )
        return weighted / 10.0  # normalize to [0, 1]
    except (ValueError, TypeError):
        return 0.0

## Demos

In [ ]:
FASHION_CONCEPTS = [
    {"concept": "sustainable streetwear with earthy tones", "target_audience": "Gen Z eco-conscious consumers"},
    {"concept": "luxury minimalist evening wear", "target_audience": "affluent millennials, 30-45"},
    {"concept": "athleisure performance collection", "target_audience": "active lifestyle enthusiasts, 25-40"},
    {"concept": "vintage-inspired workwear revival", "target_audience": "professional creatives, 28-38"},
]

def demo_baseline():
    """Show single-pass generation without evaluation."""
    print("=" * 60)
    print("BASELINE: Single-Pass Generation")
    print("=" * 60)

    generator = FashionVideoGenerator()

    for item in FASHION_CONCEPTS[:2]:
        result = generator(concept=item["concept"], target_audience=item["target_audience"])
        reward = video_reward(item, result)

        print(f"\n--- {item['concept']} ---")
        print(f"  Audience: {item['target_audience']}")
        print(f"  Image prompt: {result.image_prompt[:150]}...")
        print(f"  Motion: {result.motion_direction}")
        print(f"  Quality score: {reward:.3f}")

def demo_refine():
    """Show dspy.Refine evaluator-optimizer pattern."""
    print("\n" + "=" * 60)
    print("REFINE: Evaluator-Optimizer Pattern")
    print("=" * 60)
    print()
    print("  dspy.Refine runs the module up to N times. After each attempt,")
    print("  it evaluates the output, assigns blame to specific modules,")
    print("  and provides concrete feedback for the next attempt.")
    print()

    generator = FashionVideoGenerator()

    # Wrap with Refine: try up to 3 times, target 0.8 quality
    refined_generator = dspy.Refine(
        module=generator,
        N=3,
        reward_fn=video_reward,
        threshold=0.8,
    )

    for item in FASHION_CONCEPTS[:2]:
        # Single-pass baseline
        baseline_result = generator(concept=item["concept"], target_audience=item["target_audience"])
        baseline_score = video_reward(item, baseline_result)

        # Refined version (up to 3 attempts)
        refined_result = refined_generator(concept=item["concept"], target_audience=item["target_audience"])
        refined_score = video_reward(item, refined_result)

        print(f"\n--- {item['concept']} ---")
        print(f"  Baseline score:  {baseline_score:.3f}")
        print(f"  Refined score:   {refined_score:.3f}")
        print(f"  Delta:           {refined_score - baseline_score:+.3f}")
        print(f"  Refined prompt:  {refined_result.image_prompt[:150]}...")
        print(f"  Refined motion:  {refined_result.motion_direction}")

def demo_comparison():
    """Side-by-side: baseline vs. Refine across all concepts."""
    print("\n" + "=" * 60)
    print("COMPARISON: Baseline vs. Refine")
    print("=" * 60)

    generator = FashionVideoGenerator()
    refined = dspy.Refine(
        module=generator,
        N=3,
        reward_fn=video_reward,
        threshold=0.8,
    )

    baseline_scores = []
    refined_scores = []

    for item in FASHION_CONCEPTS:
        b_result = generator(concept=item["concept"], target_audience=item["target_audience"])
        r_result = refined(concept=item["concept"], target_audience=item["target_audience"])

        b_score = video_reward(item, b_result)
        r_score = video_reward(item, r_result)

        baseline_scores.append(b_score)
        refined_scores.append(r_score)

        print(f"\n  {item['concept'][:40]:40s}  baseline={b_score:.3f}  refined={r_score:.3f}  delta={r_score - b_score:+.3f}")

    avg_baseline = sum(baseline_scores) / len(baseline_scores)
    avg_refined = sum(refined_scores) / len(refined_scores)

    print(f"\n  {'AVERAGE':40s}  baseline={avg_baseline:.3f}  refined={avg_refined:.3f}  delta={avg_refined - avg_baseline:+.3f}")
    print()
    print("  Refine's feedback loop catches Context Clash:")
    print("  upstream prompt decisions that degrade downstream video quality.")
    print(f"  Cost: up to 3x baseline ({3} attempts × 1 LLM call + 1 eval call)")

## Run the demo

The code below mirrors the `main()` block in the source script with the argparse CLI branches removed — it runs the full demo path end-to-end.

In [ ]:
# Full demo
demo_baseline()

if not False:
    demo_refine()
    demo_comparison()